# Scaffold plank counting: YOLOv8n with leave-one-image-out
Steps: (1) upload `scaffold_data.zip` with your **corrected** labels, (2) run the leave-one-out evaluation, (3) train on all 5 and export Core ML.

In [ ]:
!pip -q install ultralytics coremltools
from google.colab import files
up = files.upload()            # choose scaffold_data.zip (with corrected labels)
!unzip -oq scaffold_data.zip
!ls scaffold_data scaffold_data/images

Saving scaffold_data.zip to scaffold_data.zip
scaffold_data:
classes.txt  images  labels  preview

scaffold_data/images:
img1.png  img2.png  img3.png  img4.png	img5.png


In [ ]:
import os, shutil, time, glob, cv2, numpy as np, torch
from ultralytics import YOLO

DATA = 'scaffold_data'
NAMES = ['img1','img2','img3','img4','img5']
# Correct counts for the FRONT stacks only (image 1 and 4 are approximate)
TRUTH = {'img1':57,'img2':23,'img3':72,'img4':15,'img5':60}
# Front-stack regions (x0, x1, y0, y1); only detections whose centre falls inside are counted
ROIS = {
 'img1': [(0,85,230,447),(90,185,225,447),(188,285,215,447),(290,400,205,447)],
 'img2': [(5,120,235,447),(125,250,250,447),(240,350,285,447)],
 'img3': [(60,135,60,372),(140,218,60,372),(222,295,60,372),(300,378,60,372)],
 'img4': [(0,200,515,700),(205,480,510,700),(485,700,520,700)],
 'img5': [(10,150,25,740),(160,310,15,770),(325,480,5,785)],
}
REPEAT = 16     # each training image is repeated with different names so one epoch has enough steps

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


In [ ]:
def build_split(train_names, out):
    shutil.rmtree(out, ignore_errors=True)
    for sub in ['images/train','labels/train','images/val','labels/val']:
        os.makedirs(f'{out}/{sub}')
    for n in train_names:
        for r in range(REPEAT):
            shutil.copy(f'{DATA}/images/{n}.png', f'{out}/images/train/{n}_{r}.png')
            shutil.copy(f'{DATA}/labels/{n}.txt', f'{out}/labels/train/{n}_{r}.txt')
    v = train_names[0]            # placeholder val (val is switched off during training)
    shutil.copy(f'{DATA}/images/{v}.png', f'{out}/images/val/{v}.png')
    shutil.copy(f'{DATA}/labels/{v}.txt', f'{out}/labels/val/{v}.txt')
    open(f'{out}/data.yaml','w').write(f'''path: {os.path.abspath(out)}
train: images/train
val: images/val
names: {{0: plank}}
''')
    return f'{out}/data.yaml'

def train(yaml, name, epochs=60):
    m = YOLO('yolov8n.pt')
    m.train(data=yaml, epochs=epochs, imgsz=800, batch=16, val=False, plots=False, verbose=False,
            mosaic=1.0, degrees=5, scale=0.5, translate=0.1, perspective=0.0005,
            hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, fliplr=0.5, flipud=0.0,
            project='runs', name=name, exist_ok=True)
    return YOLO(f'runs/detect/runs/{name}/weights/best.pt')

def count_front(model, name, conf=0.25):
    img = cv2.imread(f'{DATA}/images/{name}.png')
    t = time.perf_counter(); res = model.predict(img, imgsz=800, conf=conf, iou=0.5, verbose=False)[0]
    ms = (time.perf_counter()-t)*1000
    boxes = res.boxes.xyxy.cpu().numpy(); keep=[]
    for b in boxes:
        cx, cy = (b[0]+b[2])/2, (b[1]+b[3])/2
        if any(x0<=cx<=x1 and y0<=cy<=y1 for x0,x1,y0,y1 in ROIS[name]): keep.append(b)
    vis = img.copy()
    for b in keep: cv2.rectangle(vis,(int(b[0]),int(b[1])),(int(b[2]),int(b[3])),(0,0,255),1)
    os.makedirs('results', exist_ok=True); cv2.imwrite(f'results/{name}_pred.png', vis)
    return len(keep), len(boxes), ms

In [ ]:
rows = []
for held in NAMES:
    yaml = build_split([n for n in NAMES if n != held], f'split_{held}')

    # Use the train function directly as it is now configured to load the correct path
    model = train(yaml, f'loo_{held}')

    count_front(model, held)                       # warm-up
    n, n_all, ms = count_front(model, held)
    rows.append((held, TRUTH[held], n, n - TRUTH[held], round(ms,1)))
    print(rows[-1])

print('\nimage | correct | predicted (front stacks) | error | inference ms (' + ('GPU' if torch.cuda.is_available() else 'CPU') + ')')
for r in rows: print(*r, sep=' | ')
print('mean absolute error:', np.mean([abs(r[3]) for r in rows]))

Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split_img1/data.yaml, degrees=5, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=60, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=loo_img1, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, 

In [ ]:
# Final model: train on all 5 images, export for iOS
yaml = build_split(NAMES, 'split_all')
final = train(yaml, 'final', epochs=80)
final.export(format='coreml', imgsz=800, nms=True)    # produces runs/final/weights/best.mlpackage
!ls runs/final/weights

Ultralytics 8.4.203 🚀 Python-3.13.16 torch-2.11.0+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=split_all/data.yaml, degrees=5, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, imgsz=800, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=final, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, over

Running MIL default pipeline:  12%|█▏        | 11/95 [00:00<00:00, 107.33 passes/s]/usr/local/lib/python3.13/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '847', of the source model, has been renamed to 'var_847' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
/usr/local/lib/python3.13/dist-packages/coremltools/converters/mil/mil/passes/defs/preprocess.py:273: UserWarning: Output, '849', of the source model, has been renamed to 'var_849' in the Core ML model.
  warnings.warn(msg.format(var.name, new_name))
Running MIL backend_mlprogram pipeline: 100%|██████████| 12/12 [00:00<00:00, 82.44 passes/s]


CoreML: starting pipeline with coremltools 9.0...
CoreML: pipeline success
CoreML: export success ✅ 21.6s, saved as 'runs/detect/runs/final/weights/best.mlpackage' (6.2 MB)

Export complete (22.0s)
Results saved to /content/runs/detect/runs/final/weights/best.mlpackage
Predict:         yolo predict task=detect model=runs/detect/runs/final/weights/best.mlpackage imgsz=800 
Validate:        yolo val task=detect model=runs/detect/runs/final/weights/best.mlpackage imgsz=800 data=split_all/data.yaml 
Visualize:       https://netron.app
ls: cannot access 'runs/final/weights': No such file or directory


In [ ]:
# CPU inference time (closer to a phone than a T4); run on a CPU runtime or force device='cpu'
m = YOLO('runs/detect/runs/final/weights/best.pt'); img = cv2.imread(f'{DATA}/images/img3.png')
m.predict(img, imgsz=800, device='cpu', verbose=False)
ts=[]
for _ in range(10):
    t=time.perf_counter(); m.predict(img, imgsz=800, device='cpu', verbose=False); ts.append((time.perf_counter()-t)*1000)
print('CPU ms per image (median of 10):', round(float(np.median(ts)),1))
!zip -qr outputs.zip results runs/detect/runs/final/weights
from google.colab import files; files.download('outputs.zip')

CPU ms per image (median of 10): 174.5


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
   m = YOLO('runs/detect/runs/loo_img3/weights/best.pt')
   for c in (0.1, 0.15, 0.25, 0.4):
       print(c, count_front(m, 'img3', conf=c)[0])

0.1 65
0.15 59
0.25 44
0.4 16


In [ ]:
for n in ['img1','img2','img3','img4','img5']:
    m = YOLO(f'runs/detect/runs/loo_{n}/weights/best.pt')
    print(n, TRUTH[n], [count_front(m, n, conf=c)[0] for c in (0.1, 0.15, 0.25)])

img1 57 [59, 57, 55]
img2 23 [23, 21, 14]
img3 72 [65, 59, 44]
img4 15 [19, 15, 13]
img5 60 [66, 61, 61]
